In [1]:
#Imports & Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load the files
ratings = pd.read_csv('ratings.csv')
movies  = pd.read_csv('movies.csv')

print(f"Shape: {ratings.shape}")
print(ratings.head())
print(ratings.dtypes)
print(f"Shape: {movies.shape}")
print(movies.head())
print("Ratings NaN:", ratings.isnull().sum().to_dict())
print("Movies NaN:",  movies.isnull().sum().to_dict())
print(ratings['rating'].describe())
print()

# Convert timestamp to readable datetime
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
print(f"Earliest rating: {ratings['datetime'].min()}")
print(f"Latest rating:   {ratings['datetime'].max()}")

Shape: (100836, 4)
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
Shape: (9742, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
Ratings NaN: {'userId': 0, 'movieId': 0, 'r

In [2]:
#Preprocessing & EDA

dupes = ratings.duplicated(subset=['userId', 'movieId'])
print(f"Duplicate (userId, movieId) pairs: {dupes.sum()}")
ratings_per_user  = ratings.groupby('userId').size()
ratings_per_movie = ratings.groupby('movieId').size()
print(f"\nRatings per user  — min: {ratings_per_user.min()},  max: {ratings_per_user.max()},  median: {ratings_per_user.median()}")
print(f"Ratings per movie — min: {ratings_per_movie.min()}, max: {ratings_per_movie.max()}, median: {ratings_per_movie.median()}")
n_users  = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
sparsity = 1 - (len(ratings) / (n_users * n_movies))
print(f"\nUsers: {n_users}, Movies: {n_movies}")
print(f"Matrix sparsity: {sparsity:.4%}")

cutoff_ts = ratings['timestamp'].quantile(0.80)
cutoff_dt = pd.to_datetime(cutoff_ts, unit='s')
print(f"\nTemporal split cutoff (80th percentile): {cutoff_dt}")
print(f"  Train (before cutoff): {(ratings['timestamp'] <= cutoff_ts).sum()}")
print(f"  Test  (after cutoff):  {(ratings['timestamp'] >  cutoff_ts).sum()}")

Duplicate (userId, movieId) pairs: 0

Ratings per user  — min: 20,  max: 2698,  median: 70.5
Ratings per movie — min: 1, max: 329, median: 3.0

Users: 610, Movies: 9724
Matrix sparsity: 98.3000%

Temporal split cutoff (80th percentile): 2016-03-22 08:26:11
  Train (before cutoff): 80669
  Test  (after cutoff):  20167


In [3]:
# data splits

from sklearn.model_selection import train_test_split

# Random 80/20 split
train_random, test_random = train_test_split(
    ratings,
    test_size=0.20,
    random_state=42
)
print(f"Train: {train_random.shape}, Test: {test_random.shape}")
print(f"Train rating mean: {train_random['rating'].mean():.4f}")
print(f"Test  rating mean: {test_random['rating'].mean():.4f}")

# Temporal split
cutoff_ts = ratings['timestamp'].quantile(0.80)

train_temporal = ratings[ratings['timestamp'] <= cutoff_ts].copy()
test_temporal  = ratings[ratings['timestamp'] >  cutoff_ts].copy()

print(f"Train: {train_temporal.shape}, Test: {test_temporal.shape}")
print(f"Train date range: {train_temporal['datetime'].min()} → {train_temporal['datetime'].max()}")
print(f"Test  date range: {test_temporal['datetime'].min()} → {test_temporal['datetime'].max()}")
print(f"Train rating mean: {train_temporal['rating'].mean():.4f}")
print(f"Test  rating mean: {test_temporal['rating'].mean():.4f}")

# Cold start check for temporal
# Users/movies in test that never appear in train
test_users_unseen  = set(test_temporal['userId'])  - set(train_temporal['userId'])
test_movies_unseen = set(test_temporal['movieId']) - set(train_temporal['movieId'])
print(f"\nTemporal split — cold start:")
print(f"  Users  in test but not in train: {len(test_users_unseen)}")
print(f"  Movies in test but not in train: {len(test_movies_unseen)}")

Train: (80668, 5), Test: (20168, 5)
Train rating mean: 3.5026
Test  rating mean: 3.4975
Train: (80669, 5), Test: (20167, 5)
Train date range: 1996-03-29 18:36:55 → 2016-03-22 08:26:11
Test  date range: 2016-03-22 08:27:17 → 2018-09-24 14:27:30
Train rating mean: 3.5085
Test  rating mean: 3.4738

Temporal split — cold start:
  Users  in test but not in train: 88
  Movies in test but not in train: 1857


In [4]:
# Filtering and Matrices

# Filter cold start from temporal test set
known_users  = set(train_temporal['userId'])
known_movies = set(train_temporal['movieId'])

test_temporal_filtered = test_temporal[
    test_temporal['userId'].isin(known_users) &
    test_temporal['movieId'].isin(known_movies)
].copy()

dropped = len(test_temporal) - len(test_temporal_filtered)
print(f"Temporal test dropped {dropped} cold-start rows")
print(f"Temporal test after filtering: {test_temporal_filtered.shape}")

#random split cold start verify
random_users_unseen  = set(test_random['userId'])  - set(train_random['userId'])
random_movies_unseen = set(test_random['movieId']) - set(train_random['movieId'])
print(f"\nRandom split cold start — users: {len(random_users_unseen)}, movies: {len(random_movies_unseen)}")

# User movie rating matrix
def build_matrix(df):
    return df.pivot_table(index='userId', columns='movieId', values='rating')

train_random_matrix   = build_matrix(train_random)
train_temporal_matrix = build_matrix(train_temporal)

print(f"\nRandom train matrix shape:   {train_random_matrix.shape}")
print(f"Temporal train matrix shape: {train_temporal_matrix.shape}")

# Nans with zeros
# Mean-center by subtracting each user's mean rating
def mean_center(matrix):
    user_means = matrix.mean(axis=1)
    centered   = matrix.sub(user_means, axis=0).fillna(0)
    return centered, user_means

random_centered,   random_user_means   = mean_center(train_random_matrix)
temporal_centered, temporal_user_means = mean_center(train_temporal_matrix)
print(f"Random   user mean range: {random_user_means.min():.2f} – {random_user_means.max():.2f}")
print(f"Temporal user mean range: {temporal_user_means.min():.2f} – {temporal_user_means.max():.2f}")

Temporal test dropped 18857 cold-start rows
Temporal test after filtering: (1310, 5)

Random split cold start — users: 0, movies: 741

Random train matrix shape:   (610, 8983)
Temporal train matrix shape: (522, 7867)
Random   user mean range: 1.33 – 5.00
Temporal user mean range: 1.27 – 5.00


In [7]:
# SVD

from numpy.linalg import svd as numpy_svd

# Same cold start filtering
known_movies_random = set(train_random['movieId'])
test_random_filtered = test_random[
    test_random['movieId'].isin(known_movies_random)
].copy()
print(f"Random test after cold-start filter: {test_random_filtered.shape}")

# decomoposition
def run_svd(centered_matrix, n_components=50):
    U, sigma, Vt = numpy_svd(centered_matrix.values, full_matrices=False)
    # Truncate to k components
    U_k     = U[:, :n_components]
    sigma_k = np.diag(sigma[:n_components])
    Vt_k    = Vt[:n_components, :]
    # Reconstruct approximated matrix
    reconstructed = np.dot(U_k, np.dot(sigma_k, Vt_k))
    return pd.DataFrame(reconstructed,
                        index=centered_matrix.index,
                        columns=centered_matrix.columns)

print("\SVD Run on random split: ")
random_reconstructed   = run_svd(random_centered,   n_components=50)

print("SVD Run on temporal split")
temporal_reconstructed = run_svd(temporal_centered, n_components=50)

# prediction
def predict_rating(reconstructed_matrix, user_means, userId, movieId):
    if userId not in reconstructed_matrix.index:
        return None
    if movieId not in reconstructed_matrix.columns:
        return None
    pred = reconstructed_matrix.loc[userId, movieId] + user_means[userId]
    # Clip to valid rating range
    return float(np.clip(pred, 0.5, 5.0))

# check
sample = test_random_filtered.iloc[0]
pred   = predict_rating(random_reconstructed, random_user_means,
                        sample['userId'], sample['movieId'])
print(f"\nCheck — userId: {sample['userId']}, movieId: {sample['movieId']}")
print(f"  Actual: {sample['rating']},  Predicted: {pred:.4f}")

<>:25: SyntaxWarning: invalid escape sequence '\S'
<>:25: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_211/1726934527.py:25: SyntaxWarning: invalid escape sequence '\S'
  print("\SVD Run on random split: ")


Random test after cold-start filter: (19355, 5)
\SVD Run on random split: 
SVD Run on temporal split

Check — userId: 432, movieId: 77866
  Actual: 4.5,  Predicted: 3.6115


In [8]:
# Eval

from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(test_df, reconstructed_matrix, user_means, split_name):
    actuals    = []
    preds      = []
    skipped    = 0

    for _, row in test_df.iterrows():
        p = predict_rating(reconstructed_matrix, user_means,
                           int(row['userId']), int(row['movieId']))
        if p is None:
            skipped += 1
            continue
        actuals.append(row['rating'])
        preds.append(p)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)

    print(f"{split_name}")
    print(f"Predictions made: {len(preds)},  Skipped: {skipped}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print()
    return rmse, mae

# Evaluate both splits
rmse_r, mae_r = evaluate(test_random_filtered,   random_reconstructed,
                         random_user_means,   "Random Split (k=50)")

rmse_t, mae_t = evaluate(test_temporal_filtered, temporal_reconstructed,
                         temporal_user_means, "Temporal Split (k=50)")

# Compare across k values
for k in [10, 25, 50, 100, 150]:
    recon = run_svd(random_centered, n_components=k)
    actuals, preds = [], []
    for _, row in test_random_filtered.iterrows():
        p = predict_rating(recon, random_user_means, int(row['userId']), int(row['movieId']))
        if p is not None:
            actuals.append(row['rating'])
            preds.append(p)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    print(f"  k={k:>3}  RMSE: {rmse:.4f}  MAE: {mae:.4f}")

Random Split (k=50)
Predictions made: 19355,  Skipped: 0
RMSE: 0.9304
MAE:  0.7192

Temporal Split (k=50)
Predictions made: 1310,  Skipped: 0
RMSE: 1.0459
MAE:  0.7536

  k= 10  RMSE: 0.9184  MAE: 0.7093
  k= 25  RMSE: 0.9224  MAE: 0.7120
  k= 50  RMSE: 0.9304  MAE: 0.7192
  k=100  RMSE: 0.9396  MAE: 0.7292
  k=150  RMSE: 0.9440  MAE: 0.7333


In [10]:
# Final eval

best_k = 10

# Rebuild with best k
random_reconstructed_best   = run_svd(random_centered,   n_components=best_k)
temporal_reconstructed_best = run_svd(temporal_centered, n_components=best_k)

print(f"final result (k={best_k})\n")
rmse_r, mae_r = evaluate(test_random_filtered,   random_reconstructed_best,
                         random_user_means,   f"Random Split   (k={best_k})")
rmse_t, mae_t = evaluate(test_temporal_filtered, temporal_reconstructed_best,
                         temporal_user_means, f"Temporal Split (k={best_k})")

summary = pd.DataFrame({
    'Split':      ['Random', 'Temporal'],
    'k':          [best_k, best_k],
    'Test Size':  [len(test_random_filtered), len(test_temporal_filtered)],
    'RMSE':       [rmse_r, rmse_t],
    'MAE':        [mae_r,  mae_t]
})
print(summary.to_string(index=False))

#Baseline comparison (predict global mean for everything)
global_mean = train_random['rating'].mean()
for name, test_df in [("Random", test_random_filtered), ("Temporal", test_temporal_filtered)]:
    baseline_preds   = [global_mean] * len(test_df)
    baseline_actuals = test_df['rating'].tolist()
    b_rmse = np.sqrt(mean_squared_error(baseline_actuals, baseline_preds))
    b_mae  = mean_absolute_error(baseline_actuals, baseline_preds)
    print(f"  {name} — RMSE: {b_rmse:.4f}  MAE: {b_mae:.4f}")

print(f"\nGlobal mean used as baseline: {global_mean:.4f}")

final result (k=10)

Random Split   (k=10)
Predictions made: 19355,  Skipped: 0
RMSE: 0.9184
MAE:  0.7093

Temporal Split (k=10)
Predictions made: 1310,  Skipped: 0
RMSE: 1.0404
MAE:  0.7493

   Split  k  Test Size     RMSE      MAE
  Random 10      19355 0.918380 0.709250
Temporal 10       1310 1.040421 0.749259
  Random — RMSE: 1.0450  MAE: 0.8295
  Temporal — RMSE: 1.1219  MAE: 0.8622

Global mean used as baseline: 3.5026


In [11]:
# All metrics

from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLD = 3.5  # ratings >= 3.5 = liked (1), below = not liked (0)

def binary_metrics(test_df, reconstructed_matrix, user_means, split_name):
    actuals = []
    preds   = []

    for _, row in test_df.iterrows():
        p = predict_rating(reconstructed_matrix, user_means,
                           int(row['userId']), int(row['movieId']))
        if p is None:
            continue
        actuals.append(1 if row['rating'] >= THRESHOLD else 0)
        preds.append(1 if p >= THRESHOLD else 0)

    precision = precision_score(actuals, preds, zero_division=0)
    recall    = recall_score(actuals, preds, zero_division=0)
    f1        = f1_score(actuals, preds, zero_division=0)

    print(f"{split_name} (threshold={THRESHOLD})")
    print(f"Total predictions: {len(preds)}")
    print(f"Actual positives:  {sum(actuals)}  ({100*sum(actuals)/len(actuals):.1f}%)")
    print(f"Predicted positives: {sum(preds)} ({100*sum(preds)/len(preds):.1f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print()
    return precision, recall, f1

p_r, rec_r, f1_r = binary_metrics(test_random_filtered,   random_reconstructed_best,
                                   random_user_means,   "Random Split   (k=10)")

p_t, rec_t, f1_t = binary_metrics(test_temporal_filtered, temporal_reconstructed_best,
                                   temporal_user_means, "Temporal Split (k=10)")

for name, test_df in [("Random", test_random_filtered), ("Temporal", test_temporal_filtered)]:
    actuals = [1 if r >= THRESHOLD else 0 for r in test_df['rating']]
    preds   = [1 if global_mean >= THRESHOLD else 0] * len(test_df)
    print(f"  {name} — Precision: {precision_score(actuals, preds, zero_division=0):.4f}  "
          f"Recall: {recall_score(actuals, preds, zero_division=0):.4f}  "
          f"F1: {f1_score(actuals, preds, zero_division=0):.4f}")

Random Split   (k=10) (threshold=3.5)
Total predictions: 19355
Actual positives:  11904  (61.5%)
Predicted positives: 10475 (54.1%)
Precision: 0.7699
Recall:    0.6775
F1-score:  0.7208

Temporal Split (k=10) (threshold=3.5)
Total predictions: 1310
Actual positives:  883  (67.4%)
Predicted positives: 776 (59.2%)
Precision: 0.7951
Recall:    0.6988
F1-score:  0.7438

  Random — Precision: 0.6150  Recall: 1.0000  F1: 0.7616
  Temporal — Precision: 0.6740  Recall: 1.0000  F1: 0.8053


In [12]:
# Double mean-centering (subtract user bias AND item/movie bias) to try and improve the shite performance

def double_mean_center(matrix):
    global_mean = matrix.stack().mean()
    user_means  = matrix.mean(axis=1)
    item_means  = matrix.mean(axis=0)

    # User bias and item bias relative to global mean
    user_bias = user_means - global_mean
    item_bias = item_means - global_mean

    # Subtract both biases
    centered = matrix.copy()
    centered = centered.sub(user_bias, axis=0)
    centered = centered.sub(item_bias, axis=1)
    centered = centered.fillna(0)

    return centered, global_mean, user_bias, item_bias

def predict_double(reconstructed, global_mean, user_bias, item_bias, userId, movieId):
    if userId not in reconstructed.index or movieId not in reconstructed.columns:
        return None
    pred = (reconstructed.loc[userId, movieId]
            + global_mean
            + user_bias[userId]
            + item_bias[movieId])
    return float(np.clip(pred, 0.5, 5.0))

# Build double-centered matrices
r_centered2, r_gmean, r_ubias, r_ibias = double_mean_center(train_random_matrix)
t_centered2, t_gmean, t_ubias, t_ibias = double_mean_center(train_temporal_matrix)

# Run SVD with best k
r_recon2 = run_svd(r_centered2, n_components=10)
t_recon2 = run_svd(t_centered2, n_components=10)

# Eval
for name, test_df, recon, gmean, ubias, ibias in [
    ("Random",   test_random_filtered,   r_recon2, r_gmean, r_ubias, r_ibias),
    ("Temporal", test_temporal_filtered, t_recon2, t_gmean, t_ubias, t_ibias)
]:
    actuals, preds = [], []
    for _, row in test_df.iterrows():
        p = predict_double(recon, gmean, ubias, ibias, int(row['userId']), int(row['movieId']))
        if p is not None:
            actuals.append(row['rating'])
            preds.append(p)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    bin_actual = [1 if a >= 3.5 else 0 for a in actuals]
    bin_pred   = [1 if p >= 3.5 else 0 for p in preds]
    precision  = precision_score(bin_actual, bin_pred, zero_division=0)
    recall     = recall_score(bin_actual, bin_pred, zero_division=0)
    f1         = f1_score(bin_actual, bin_pred, zero_division=0)

    print(f"  {name}: RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

  Random: RMSE=1.1685  MAE=0.8981  P=0.7084  R=0.8727  F1=0.7820
  Temporal: RMSE=1.1611  MAE=0.8590  P=0.7683  R=0.8675  F1=0.8149


In [15]:
# Funk SVD (SGD-based matrix factorization)

class FunkSVD:
    def __init__(self, n_factors=50, n_epochs=30, lr=0.005, reg=0.02, random_state=42):
        self.n_factors   = n_factors
        self.n_epochs    = n_epochs
        self.lr          = lr
        self.reg         = reg
        self.random_state = random_state

    def fit(self, train_df):
        np.random.seed(self.random_state)
        self.global_mean = train_df['rating'].mean()

        # Index mappings
        self.user_idx = {u: i for i, u in enumerate(train_df['userId'].unique())}
        self.item_idx = {m: i for i, m in enumerate(train_df['movieId'].unique())}
        n_users = len(self.user_idx)
        n_items = len(self.item_idx)

        # Initialize latent factors and biases
        self.P  = np.random.normal(0, 0.1, (n_users, self.n_factors))  # user factors
        self.Q  = np.random.normal(0, 0.1, (n_items, self.n_factors))  # item factors
        self.bu = np.zeros(n_users)   # user biases
        self.bi = np.zeros(n_items)   # item biases

        # SGD
        records = train_df[['userId', 'movieId', 'rating']].values
        for epoch in range(self.n_epochs):
            np.random.shuffle(records)
            for userId, movieId, rating in records:
                u = self.user_idx.get(userId)
                i = self.item_idx.get(movieId)
                if u is None or i is None:
                    continue
                pred = self._predict_raw(u, i)
                err  = rating - pred

                # Update biases
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])

                # Update factors
                pu = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * pu        - self.reg * self.Q[i])

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{self.n_epochs} done")
        return self

    def _predict_raw(self, u, i):
        return (self.global_mean + self.bu[u] + self.bi[i]
                + np.dot(self.P[u], self.Q[i]))

    def predict(self, userId, movieId):
        u = self.user_idx.get(userId)
        i = self.item_idx.get(movieId)
        if u is None or i is None:
            return None
        return float(np.clip(self._predict_raw(u, i), 0.5, 5.0))

def eval_funk(model, test_df, split_name):
    actuals, preds = [], []
    for _, row in test_df.iterrows():
        p = model.predict(int(row['userId']), int(row['movieId']))
        if p is not None:
            actuals.append(row['rating'])
            preds.append(p)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)

    bin_a = [1 if a >= 3.5 else 0 for a in actuals]
    bin_p = [1 if p >= 3.5 else 0 for p in preds]
    prec  = precision_score(bin_a, bin_p, zero_division=0)
    rec   = recall_score(bin_a, bin_p, zero_division=0)
    f1    = f1_score(bin_a, bin_p, zero_division=0)

    print(f"  {split_name}: RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    return rmse, mae, prec, rec, f1
print("Random Split trainign")
funk_r = FunkSVD(n_factors=50, n_epochs=30).fit(train_random)
print("Temporal split training")
funk_t = FunkSVD(n_factors=50, n_epochs=30).fit(train_temporal)
eval_funk(funk_r, test_random_filtered,   "Random")
eval_funk(funk_t, test_temporal_filtered, "Temporal")

Random Split trainign
  Epoch 10/30 done
  Epoch 20/30 done
  Epoch 30/30 done
Temporal split training
  Epoch 10/30 done
  Epoch 20/30 done
  Epoch 30/30 done
  Random: RMSE=0.8769  MAE=0.6704  P=0.7982  R=0.6972  F1=0.7443
  Temporal: RMSE=0.9310  MAE=0.6707  P=0.8141  R=0.7984  F1=0.8062


(np.float64(0.9310279934531516),
 0.6706553305489448,
 0.8140877598152425,
 0.79841449603624,
 0.8061749571183533)

In [16]:
#Grid search over Funk SVD hyperparameters (random split only for speed)

from itertools import product

param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs':  [20, 40],
    'lr':        [0.005, 0.01],
    'reg':       [0.02, 0.05]
}

best_rmse   = float('inf')
best_params = {}
results     = []
combos = list(product(
    param_grid['n_factors'],
    param_grid['n_epochs'],
    param_grid['lr'],
    param_grid['reg']
))

print(f"Testing {len(combos)} combinations\n")

for n_factors, n_epochs, lr, reg in combos:
    model = FunkSVD(n_factors=n_factors, n_epochs=n_epochs,
                    lr=lr, reg=reg, random_state=42)
    model.fit(train_random)

    actuals, preds = [], []
    for _, row in test_random_filtered.iterrows():
        p = model.predict(int(row['userId']), int(row['movieId']))
        if p is not None:
            actuals.append(row['rating'])
            preds.append(p)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    results.append((n_factors, n_epochs, lr, reg, rmse, mae))

    if rmse < best_rmse:
        best_rmse   = rmse
        best_params = dict(n_factors=n_factors, n_epochs=n_epochs, lr=lr, reg=reg)
        best_model_r = model
        print(f"  New best — factors={n_factors}, epochs={n_epochs}, "
              f"lr={lr}, reg={reg} → RMSE={rmse:.4f}")
results_df = pd.DataFrame(results,
    columns=['n_factors', 'n_epochs', 'lr', 'reg', 'RMSE', 'MAE'])
print(results_df.sort_values('RMSE').head(5).to_string(index=False))
print(f"\nBest params: {best_params}")
print(f"Best RMSE on random split: {best_rmse:.4f}")

Testing 24 combinations

  Epoch 10/20 done
  Epoch 20/20 done
  New best — factors=20, epochs=20, lr=0.005, reg=0.02 → RMSE=0.8745
  Epoch 10/20 done
  Epoch 20/20 done
  New best — factors=20, epochs=20, lr=0.005, reg=0.05 → RMSE=0.8744
  Epoch 10/20 done
  Epoch 20/20 done
  Epoch 10/20 done
  Epoch 20/20 done
  New best — factors=20, epochs=20, lr=0.01, reg=0.05 → RMSE=0.8648
  Epoch 10/40 done
  Epoch 20/40 done
  Epoch 30/40 done
  Epoch 40/40 done
  Epoch 10/40 done
  Epoch 20/40 done
  Epoch 30/40 done
  Epoch 40/40 done
  New best — factors=20, epochs=40, lr=0.005, reg=0.05 → RMSE=0.8645
  Epoch 10/40 done
  Epoch 20/40 done
  Epoch 30/40 done
  Epoch 40/40 done
  Epoch 10/40 done
  Epoch 20/40 done
  Epoch 30/40 done
  Epoch 40/40 done
  Epoch 10/20 done
  Epoch 20/20 done
  Epoch 10/20 done
  Epoch 20/20 done
  Epoch 10/20 done
  Epoch 20/20 done
  Epoch 10/20 done
  Epoch 20/20 done
  New best — factors=50, epochs=20, lr=0.01, reg=0.05 → RMSE=0.8642
  Epoch 10/40 done
  Epo

In [17]:
# Ensemble (truncated SVD + best Funk SVD) + KNN collaborative filtering

# Train best Funk SVD on temporal split too
best_model_t = FunkSVD(n_factors=100, n_epochs=40, lr=0.01, reg=0.05, random_state=42)
best_model_t.fit(train_temporal)
print("Done.\n")

#Ensemble: average truncated SVD + Funk SVD predictions
def ensemble_predict(userId, movieId, trunc_recon, user_means, funk_model, w_trunc=0.4, w_funk=0.6):
    p_trunc = predict_rating(trunc_recon, user_means, userId, movieId)
    p_funk  = funk_model.predict(userId, movieId)
    if p_trunc is None and p_funk is None:
        return None
    if p_trunc is None:
        return p_funk
    if p_funk is None:
        return p_trunc
    return float(np.clip(w_trunc * p_trunc + w_funk * p_funk, 0.5, 5.0))

# KNN collaborative filtering (user-based)
from sklearn.metrics.pairwise import cosine_similarity

def build_knn_model(train_matrix, k=20):
    filled    = train_matrix.fillna(0)
    sim_matrix = cosine_similarity(filled)
    sim_df     = pd.DataFrame(sim_matrix,
                               index=train_matrix.index,
                               columns=train_matrix.index)
    return sim_df

def knn_predict(train_matrix, sim_df, user_means, userId, movieId, k=20):
    if userId not in train_matrix.index:
        return None
    if movieId not in train_matrix.columns:
        return None

    # Get k most similar users who rated this movie
    sim_scores = sim_df[userId].drop(userId)
    rated_mask = train_matrix[movieId].notna()
    sim_scores = sim_scores[rated_mask]

    if len(sim_scores) == 0:
        return None

    top_k   = sim_scores.nlargest(k)
    ratings = train_matrix.loc[top_k.index, movieId]
    biases  = user_means[top_k.index]
    # Weighted average of neighbor ratings (mean-centered)
    numer   = (top_k * (ratings - biases)).sum()
    denom   = top_k.abs().sum()
    if denom == 0:
        return None
    pred = user_means[userId] + numer / denom
    return float(np.clip(pred, 0.5, 5.0))

# Build KNN models
sim_r = build_knn_model(train_random_matrix)
sim_t = build_knn_model(train_temporal_matrix)
print("Done.\n")

# Evaluate
def full_eval(test_df, actuals_fn, preds_fn, split_name):
    actuals, preds = [], []
    for _, row in test_df.iterrows():
        p = preds_fn(int(row['userId']), int(row['movieId']))
        if p is not None:
            actuals.append(row['rating'])
            preds.append(p)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    bin_a = [1 if a >= 3.5 else 0 for a in actuals]
    bin_p = [1 if p >= 3.5 else 0 for p in preds]
    prec  = precision_score(bin_a, bin_p, zero_division=0)
    rec   = recall_score(bin_a, bin_p, zero_division=0)
    f1    = f1_score(bin_a, bin_p, zero_division=0)
    print(f"  {split_name}: RMSE={rmse:.4f}  MAE={mae:.4f}  "
          f"P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    return rmse, mae, prec, rec, f1

full_eval(test_random_filtered,   None,
          lambda u, m: best_model_r.predict(u, m), "Random")
full_eval(test_temporal_filtered, None,
          lambda u, m: best_model_t.predict(u, m), "Temporal")

full_eval(test_random_filtered, None,
          lambda u, m: ensemble_predict(u, m, random_reconstructed_best,
                                        random_user_means, best_model_r), "Random")
full_eval(test_temporal_filtered, None,
          lambda u, m: ensemble_predict(u, m, temporal_reconstructed_best,
                                        temporal_user_means, best_model_t), "Temporal")

full_eval(test_random_filtered, None,
          lambda u, m: knn_predict(train_random_matrix, sim_r,
                                   random_user_means, u, m), "Random")
full_eval(test_temporal_filtered, None,
          lambda u, m: knn_predict(train_temporal_matrix, sim_t,
                                   temporal_user_means, u, m), "Temporal")

  Epoch 10/40 done
  Epoch 20/40 done
  Epoch 30/40 done
  Epoch 40/40 done
Done.

Done.

  Random: RMSE=0.8596  MAE=0.6549  P=0.8035  R=0.7124  F1=0.7552
  Temporal: RMSE=0.9219  MAE=0.6654  P=0.8082  R=0.8018  F1=0.8050
  Random: RMSE=0.8615  MAE=0.6585  P=0.8022  R=0.7180  F1=0.7578
  Temporal: RMSE=0.9421  MAE=0.6722  P=0.8159  R=0.7882  F1=0.8018
  Random: RMSE=0.8914  MAE=0.6746  P=0.7904  R=0.7315  F1=0.7598
  Temporal: RMSE=0.9980  MAE=0.7121  P=0.8082  R=0.8018  F1=0.8050


(np.float64(0.9980130135376294),
 0.7121368768098524,
 0.8082191780821918,
 0.8018120045300113,
 0.8050028425241614)